# Hybrid Search for Agentic Systems

> Based on: [How to Build Agentic RAG with Hybrid Search — Towards Data Science](https://towardsdatascience.com/how-to-build-agentic-rag-with-hybrid-search/)

## 1. Imports
We start by loading our libraries for keyword matching, vector indexing, and calling our chosen language model.

In [1]:
import math
import json
import numpy as np
import os
import re
import warnings

warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TQDM_DISABLE"] = "1"

from collections import defaultdict
from dataclasses import dataclass, field
from typing import Any, Callable

# BM25
from rank_bm25 import BM25Okapi

# Dense embeddings + FAISS ANN index
from sentence_transformers import SentenceTransformer
import faiss

# LLM
import anthropic

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

print("All imports OK")

def print_results(label: str, get_results: Callable[[], list[tuple[int, float]]]) -> None:
    print(f"\n{'─'*60}")
    print(f"  {label}")
    print('─'*60)
    results = get_results()
    for doc_id, score in results:
        print(f"  [{doc_id:02d}] score={score:.4f}  |  {DOCS[doc_id]['text']}")
    pass


All imports OK


## 2. Define Corpus and Input Query
Setting up our dataset representing banking knowledgebase articles, and defining the input issue we want to troubleshoot.

In [2]:
# Sample generated corpus based on Gemini's training data so it may not be factually accurate.
# In real corpora, it is preferable to have clearer documents.
# This dataset is meant to provide a challenging scenario with a small corpus so the differences between the methods are clear.
DOCS = [
    {"id": 0, "text": "Domestic bank transfers (ACH) normally complete within 1-2 business days."},
    {"id": 1, "text": "Wire transfers are usually processed on the same business day if submitted before the cutoff time."},
    {"id": 2, "text": "Checks generally take 2-5 business days to clear, depending on the issuing bank and amount."},
    {"id": 3, "text": "When a cheque is deposited, funds are placed on a temporary hold to ensure they can be collected from the paying bank."},
    {"id": 4, "text": "If details shows PENDING_KB200 this means clearance is still in progress."},
    {"id": 5, "text": "Status: COMPLETED indicates that funds from a transaction have successfully settled and are fully available."},
    {"id": 6, "text": "Status: RETURNED means the paying bank rejected the check, often due to insufficient funds (NSF)."},
    {"id": 7, "text": "For checks over $5,000, extended holds of up to 7 business days may apply per Regulation CC."},
    {"id": 8, "text": "New accounts (less than 30 days old) may face up to 9 business days of hold times on deposited checks."},
    {"id": 9, "text": "International checks can take significantly longer to clear, often 4-6 weeks."},
    {"id": 10, "text": "All checks must be properly endorsed on the back to avoid processing delays."},
    {"id": 11, "text": "A valid check must include the date, payee name, numeric amount, written amount, and signature of the payer."},
    {"id": 12, "text": "Mobile deposits might have additional processing time if the image is unclear or flagged for manual review."},
    {"id": 13, "text": "Customers can view their available balance versus current balance on the online banking portal."},
    {"id": 14, "text": "Current balance includes all transactions, including those on hold, whereas available balance only includes cleared funds."},
    {"id": 15, "text": "Status: PROCESSING indicates a wire or ACH transfer has been initiated but not yet concluded."},
    {"id": 16, "text": "To clear a check, the Federal Reserve Bank often routes the physical or digital image to the payer's financial institution."},
    {"id": 17, "text": "If a queue hold is extended, the bank must provide a written notice of delayed availability to the customer."},
    {"id": 18, "text": "Cash deposits are usually available immediately upon receipt by the teller."},
    {"id": 19, "text": "For wire transfers, the required fields include the recipient's routing number, account number, name, and address."},
    {"id": 20, "text": "A cashier's check is guaranteed by the bank and typically clears faster, often next-day availability."},
    {"id": 21, "text": "Money orders are similar to cashier's checks but usually have lower maximum limits."},
    {"id": 22, "text": "Status: PENDING_VERIFICATION is used when a transaction is flagged by anti-fraud systems."},
    {"id": 23, "text": "The routing number on a check identifies the specific financial institution it is drawn from."},
    {"id": 24, "text": "Checks with discrepancies between the numeric and written amounts will be processed based on the written amount."},
    {"id": 25, "text": "Post-dated checks cannot be deposited until the date written on the check arrives."},
    {"id": 26, "text": "Stale-dated checks, which are older than 6 months, may be refused by the bank."},
    {"id": 27, "text": "ACH transfers are settled in batches throughout the day rather than instantly."},
    {"id": 28, "text": "Status: CANCELLED means the payer or payee has revoked the transaction before processing began."},
    {"id": 29, "text": "Customer support should advise clients with cheques pending clearance to wait up to 5 business days for standard checks to clear."},
    {"id": 30, "text": "Status: PENDING_KB501 refers to ACH transfers that are currently queued for processing in the next batch window."},
    {"id": 31, "text": "Status: PENDING_KB105 means a cash deposit was made at an ATM but has not yet been physically verified by a teller."},
    {"id": 32, "text": "A transaction with PENDING_KB880 is awaiting manager approval due to exceeding typical transaction limits."},
    {"id": 33, "text": "The error code PENDING_KB009 indicates a mismatch in the routing number and requires manual customer verification."},
    {"id": 34, "text": "If a transfer is marked PENDING_KB422, it signifies a potential OFAC or AML hold pending further review."},
    {"id": 35, "text": "For international incoming wires, PENDING_KB991 indicates the funds are subject to currency conversion delays."},
    {"id": 36, "text": "PENDING_KB347 is a common code for mobile check deposits waiting for imaging clear validation."},
    {"id": 37, "text": "When an account goes into overdraft but has backup protection, transactions may briefly sit in PENDING_KB611."},
    {"id": 38, "text": "Status: PENDING_KB733 reflects an internal system delay syncing data between branches and the central mainframe."},
    {"id": 39, "text": "If a customer asks about PENDING_KB244, explain it means their replacement debit card issuance fee is processing."},
    {"id": 40, "text": "Please refer to article 200 in the compliance handbook for regulatory holds on large accounts."},
    {"id": 41, "text": "Our CRM limits the display of recent transaction history to 200 records per page for performance reasons."},
    {"id": 42, "text": "A standard personal check book contains 200 checks, but business accounts can order binders with 600 or more."},
    {"id": 43, "text": "There is a $200 minimum balance required to waive the monthly maintenance fee on basic savings accounts."},
    {"id": 44, "text": "ATM withdrawals are generally capped at $200 per day for new accounts until they pass their 30-day probationary period."},
    {"id": 45, "text": "The new terminal update requires a reboot; code PENDING_KB200 indicates the patch is stalled waiting for admin to check it."},
    {"id": 46, "text": "Status PENDING_KB200 check tray means the paper jam is cleared but the tray sensor has not yet reset."},
    {"id": 47, "text": "If the website is not available try refreshing the page."},
    {"id": 48, "text": "Funds deposited via mobile app are subject to verification before being added to your available balance."},
    {"id": 49, "text": "Check processing typically occurs overnight during standard batch reconciliation windows."},
    {"id": 50, "text": "Status: HOLD means your funds are not yet cleared and cannot be withdrawn or used for purchases."},
    {"id": 51, "text": "To expedite check clearance, always ensure the written amount matches the numeric amount precisely."},
    {"id": 52, "text": "Sufficient funds must be present in your account to avoid a returned check fee of $35."},
    {"id": 53, "text": "The current status of your check deposit can be easily tracked inside the secure banking portal."},
    {"id": 54, "text": "Ensure all check endorsements on the back are correct so deposited funds can be released quickly."},
    {"id": 55, "text": "Status: COMPLETED confirms the funds have successfully transferred and are ready for use."},
    {"id": 56, "text": "Wire transfers move funds much faster than a traditional paper check, often within hours."},
    {"id": 57, "text": "The processing status of an ACH transfer will remain pending until the regional batch finally settles."},
    {"id": 58, "text": "Funds from a cashier's check are considered guaranteed and are often available the next business day."},
    {"id": 59, "text": "If a check's status is still pending, the associated funds absolutely cannot be withdrawn yet."},
    {"id": 60, "text": "Depositing an out-of-state check may delay the availability of your funds by several extra days."},
    {"id": 61, "text": "We advise checking your account status daily to closely monitor available funds and spot unauthorized transactions."},
    {"id": 62, "text": "To verify funds on large transactions, the teller may be required to call the check's issuing bank."},
    {"id": 63, "text": "Status: REJECTED heavily implies the check was flagged as fraudulent, unauthorized, or poorly imaged."},
    {"id": 64, "text": "Funds held from a pending check will immediately reflect in your current balance but not in your available balance."},
    {"id": 65, "text": "Please bring a valid, government-issued ID to the branch if you wish to cash a check and receive physical funds."},
    {"id": 66, "text": "A stop payment order on a check will immediately change its processing status to cancelled."},
    {"id": 67, "text": "Always carefully confirm the check date to avoid any annoying issues with funds settlement and maturity."},
    {"id": 68, "text": "Status: PENDING processing usually resolves unconditionally within 24 hours for straightforward internal account transfers."},
    {"id": 69, "text": "Mobile check images absolutely must capture all four corners to securely process and release the funds."},
    {"id": 70, "text": "Joint account holders share equal access to all securely deposited funds and full check history logs."},
    {"id": 71, "text": "Status: PENDING_VERIFICATION clearly indicates a manual compliance review of the submitted check."},
    {"id": 72, "text": "Processing a standard counter check may require additional time for strict funds verification."},
    {"id": 73, "text": "Electronic direct deposits always provide much faster access to cleared funds than a physical paper check."},
    {"id": 74, "text": "If an submitted check is deemed suspicious, related funds will be frozen indefinitely for investigation."},
    {"id": 75, "text": "Closely monitor your transaction status and funds if you have any reason to suspect active fraud."},
    {"id": 76, "text": "Check washing is a very common financial crime that steals unprotected funds directly from your account."},
    {"id": 77, "text": "The central clearinghouse status officially determines exactly when deposited funds are ultimately settled."},
    {"id": 78, "text": "When a check clears smoothly, the internal status updates to cleared and the associated funds are fully unlocked."},
    {"id": 79, "text": "Sufficient, fully settled funds are completely necessary to request and issue a certified check."},
    {"id": 80, "text": "A temporary starter check may unfortunately not be accepted by all local merchants for services or goods."},
    {"id": 81, "text": "Funds from most international checks are heavily subject to incredibly long, multi-week processing times."},
    {"id": 82, "text": "Status: REVIEW actively means our dedicated fraud department is currently looking at the physical check image."},
    {"id": 83, "text": "It is best practice to warmly endorse the check 'For Deposit Only' to thoroughly protect your funds."},
    {"id": 84, "text": "Various check processing fees may automatically apply depending strictly on your current account status and tier."},
    {"id": 85, "text": "Always keep a detailed record of your written check numbers to closely track any missing funds."},
    {"id": 86, "text": "Important status updates for any submitted checks are pushed to the user's mobile app instantly via notifications."},
    {"id": 87, "text": "Third-party checks very often face significantly stricter funds availability and hold policies at our branches."},
    {"id": 88, "text": "A post-dated check fundamentally cannot actively grant an account holder access to funds prematurely."},
    {"id": 89, "text": "Regularly review your detailed monthly statement carefully to meticulously verify the status of all processed checks."},
    {"id": 90, "text": "Overdraft protection services can safely supply immediate funds if a written check unavoidably exceeds your total balance."},
    {"id": 91, "text": "A cancelled voided check is very often firmly required by HR to set up electronic direct deposit of salary funds."},
    {"id": 92, "text": "Status: RETURNED_NSF directly means there are currently insufficient funds located at the paying bank's account."},
    {"id": 93, "text": "Optimized electronic check conversion safely processes and settles funds far more rapidly than older methods."},
    {"id": 94, "text": "Any customer can securely request a physical paper copy of a cleared check for their personal records."},
    {"id": 95, "text": "Lost check replacements legally require firmly cancelling the original check and completely securing its intended funds."},
    {"id": 96, "text": "The secure transmission status of an electronic check depends solely on the automated banking network."},
    {"id": 97, "text": "Corresponding funds are deducted immediately from your account when a guaranteed cashier's check is created."},
    {"id": 98, "text": "Severe check fraud can negatively and permanently impact your broader account standing and overall status."},
    {"id": 99, "text": "Use convenient mobile alerts to strictly know exactly when incoming check funds safely become available."}
]

input_query = "Why are the funds for a check still in PENDING_KB200 status?"
print("Corpus loaded. Query:", input_query)

Corpus loaded. Query: Why are the funds for a check still in PENDING_KB200 status?


## 3. Keyword Search
Tokenizing the dataset/query and executing a sparse lexical search using Okapi BM25.

In [3]:
# Tokenise (replace specific stop words explicitly, then lowercase and split on whitespace)
STOP_WORDS = {"the", "a", "an", "for", "in", "of", "to", "is", "are", "with", "and", "or", "why", "still", "status", "this", "that", "it"}

def custom_tokenize(text: str) -> list[str]:
    # Very simple tokenization that preserves underscores
    clean_text = text.replace(",", " ").replace(".", " ").replace("?", " ").replace(";", " ").replace(":", " ").lower()
    tokens = clean_text.split()
    return [t for t in tokens if t not in STOP_WORDS]

tokenised_corpus = [custom_tokenize(doc["text"]) for doc in DOCS]
keyword_index = BM25Okapi(tokenised_corpus)

def keyword_search(query: str, top_k: int = 5) -> list[tuple[int, float]]:
    tokens = custom_tokenize(query)
    scores = keyword_index.get_scores(tokens)
    ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
    return [(doc_id, score) for doc_id, score in ranked[:top_k] if score > 0]

## 4. Semantic Search
Encoding the text corpus into dense semantic embeddings and indexing them via FAISS cosine.

In [4]:
# Build a FAISS flat index (exact cosine via inner-product on normalised vectors)
model = SentenceTransformer("all-MiniLM-L6-v2")

corpus_texts = [doc["text"] for doc in DOCS]
corpus_embeddings = model.encode(corpus_texts, normalize_embeddings=True)  # L2-normalised → dot == cosine

dim = corpus_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)  # Inner Product on unit vectors = cosine similarity
index.add(corpus_embeddings.astype("float32"))

print(f"FAISS index built: {index.ntotal} vectors of dim={dim}")

def semantic_search(query: str, top_k: int = 5) -> list[tuple[int, float]]:
    q_emb = model.encode([query], normalize_embeddings=True).astype("float32")
    scores, indices = index.search(q_emb, top_k)
    return [(int(idx), float(score)) for idx, score in zip(indices[0], scores[0]) if idx != -1]

FAISS index built: 100 vectors of dim=384


## 5. Compare Results (Keyword vs Semantic)
Running both search techniques independently to observe their different strengths resolving the problem query.

In [5]:
print(f'QUERY: "{input_query}"')
print_results("Keyword only", lambda: keyword_search(query=input_query, top_k=3))
print_results("Semantic only", lambda: semantic_search(query=input_query, top_k=3))

QUERY: "Why are the funds for a check still in PENDING_KB200 status?"

────────────────────────────────────────────────────────────
  Keyword only
────────────────────────────────────────────────────────────
  [04] score=4.1156  |  If details shows PENDING_KB200 this means clearance is still in progress.
  [45] score=3.3460  |  The new terminal update requires a reboot; code PENDING_KB200 indicates the patch is stalled waiting for admin to check it.
  [46] score=3.2302  |  Status PENDING_KB200 check tray means the paper jam is cleared but the tray sensor has not yet reset.

────────────────────────────────────────────────────────────
  Semantic only
────────────────────────────────────────────────────────────
  [31] score=0.6846  |  Status: PENDING_KB105 means a cash deposit was made at an ATM but has not yet been physically verified by a teller.
  [64] score=0.6638  |  Funds held from a pending check will immediately reflect in your current balance but not in your available balance.
 

## 6. Hybrid Fusion

In the previous cells keyword vs. sematic are compared:

- **Keyword Search (Sparse)** Excels at exact term matching (e.g. error codes, IDs, specific jargon), but struggles with synonyms or implied concepts.
- **Semantic Search (Dense)** Excels at understanding intent and concepts even when words don't match exactly, but struggles with exact identifiers.

**Hybrid search** leverages both by merging sparse and dense rank lists together, filling in their individual gaps.

These two algorithms for hybrid search are used as example:

- **Reciprocal Rank Fusion (RRF):** Combines multiple ranked lists by prioritizing items that consistently appear near the top across different searches, relying purely on ranking positions rather than scores.
- **Convex Score Combination (Weighted):** Normalizes the actual similarity scores from different search methods and combines them using a weighted average (alpha parameter).

**How they differ:** RRF ignores the underlying score numbers and uses only the rank order (making it robust when scores have drastically different scales), while the Weighted approach uses the normalized score values directly (allowing for fine-tuned control over how much weight each search method gets).


In [6]:
def reciprocal_rank_fusion(
    *ranked_lists: list[tuple[int, float]], k: int = 60
) -> list[tuple[int, float]]:
    """Merge ranked lists using RRF. Each list is [(doc_id, score), ...]."""
    rrf_scores: dict[int, float] = defaultdict(float)
    for ranked_list in ranked_lists:
        for rank, (doc_id, _score) in enumerate(ranked_list, start=1):
            rrf_scores[doc_id] += 1.0 / (k + rank)
    return sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

def minmax_normalize(scores: list[tuple[int, float]]) -> list[tuple[int, float]]:
    if not scores:
        return scores
    values = [s for _, s in scores]
    lo, hi = min(values), max(values)
    if hi == lo:
        return [(doc_id, 1.0) for doc_id, _ in scores]
    return [(doc_id, (s - lo) / (hi - lo)) for doc_id, s in scores]

def convex_combination(
    dense_results: list[tuple[int, float]],
    sparse_results: list[tuple[int, float]],
    alpha: float = 0.5,
) -> list[tuple[int, float]]:
    """Weighted score fusion. alpha=1 → pure dense; alpha=0 → pure BM25."""
    dense_norm = dict(minmax_normalize(dense_results))
    sparse_norm = dict(minmax_normalize(sparse_results))
    all_ids = set(dense_norm) | set(sparse_norm)
    combined = {
        doc_id: alpha * dense_norm.get(doc_id, 0.0) + (1 - alpha) * sparse_norm.get(doc_id, 0.0)
        for doc_id in all_ids
    }
    return sorted(combined.items(), key=lambda x: x[1], reverse=True)

def hybrid_search(
    query: str, top_k: int = 5, fusion: str = "rrf", alpha: float = 0.5
) -> list[tuple[int, float]]:
    """Run Keyword + Semantic then fuse. fusion='rrf' or 'convex'."""
    keyword_results = keyword_search(query, top_k=top_k * 2)
    semantic_results = semantic_search(query, top_k=top_k * 2)
    if fusion == "rrf":
        merged = reciprocal_rank_fusion(keyword_results, semantic_results)
    else:
        merged = convex_combination(semantic_results, keyword_results, alpha=alpha)
    return merged[:top_k]

## 7. Compare Results (All previous)
Comparing Keyword, Semantic, and Hybrid techniques head-to-head.

In [7]:
print(f'QUERY: "{input_query}"')
print_results("Keyword only", lambda: keyword_search(query=input_query, top_k=3))
print_results("Semantic only", lambda: semantic_search(query=input_query, top_k=3))
print_results("Hybrid — RRF (k=60)", lambda: hybrid_search(query=input_query, top_k=3, fusion="rrf"))
print_results("Hybrid — Convex α=0.5", lambda: hybrid_search(query=input_query, top_k=3, fusion="convex", alpha=0.5))

QUERY: "Why are the funds for a check still in PENDING_KB200 status?"

────────────────────────────────────────────────────────────
  Keyword only
────────────────────────────────────────────────────────────
  [04] score=4.1156  |  If details shows PENDING_KB200 this means clearance is still in progress.
  [45] score=3.3460  |  The new terminal update requires a reboot; code PENDING_KB200 indicates the patch is stalled waiting for admin to check it.
  [46] score=3.2302  |  Status PENDING_KB200 check tray means the paper jam is cleared but the tray sensor has not yet reset.

────────────────────────────────────────────────────────────
  Semantic only
────────────────────────────────────────────────────────────
  [31] score=0.6846  |  Status: PENDING_KB105 means a cash deposit was made at an ATM but has not yet been physically verified by a teller.
  [64] score=0.6638  |  Funds held from a pending check will immediately reflect in your current balance but not in your available balance.
 

## 8. Graph Augmented Retrieval
Extracting conceptual entities to build a relationship layout, then retrieving via graph-traversed neighborhoods.

In [8]:
# ── Step 1: Extract entities ─────────────────────────────
ENTITY_PATTERNS = re.compile(
    r'\b([A-Z][A-Z0-9_\-]{2,}|BM25|kNN|HNSW|RRF|RAG|FAISS|'
    r'PostgreSQL|MySQL|Python|NDCG|MRR)\b'
)

def extract_entities(text: str) -> list[str]:
    return list({m.group(0) for m in ENTITY_PATTERNS.finditer(text)})

entity_to_docs: dict[str, set[int]] = defaultdict(set)
doc_to_entities: dict[int, list[str]] = {}

for doc in DOCS:
    entities = extract_entities(doc["text"])
    doc_to_entities[doc["id"]] = entities
    for ent in entities:
        entity_to_docs[ent].add(doc["id"])

# ── Step 2: Build edges ──────────────────────
doc_graph: dict[int, set[int]] = defaultdict(set)

for ent, doc_ids in entity_to_docs.items():
    doc_list = list(doc_ids)
    for i in range(len(doc_list)):
        for j in range(i + 1, len(doc_list)):
            doc_graph[doc_list[i]].add(doc_list[j])
            doc_graph[doc_list[j]].add(doc_list[i])

# ── Step 3: Graph RAG ───────────────────────────────────────
def graph_rag_search(query: str, top_k: int = 5, expand_hops: int = 1, verbose: bool = True) -> list[tuple[int, float]]:
    seed_results = hybrid_search(query, top_k=top_k, fusion="rrf")
    seed_ids = {doc_id for doc_id, _ in seed_results}
    
    if verbose:
        print(f"\n[Step 1: Finding Seeds] Using Hybrid Search to find the best initial documents/clues...")
        for doc_id, score in seed_results:
            entities = doc_to_entities.get(doc_id, [])
            print(f"  ↳ Identified Seed [Doc {doc_id}] containing key entities: {entities}\n    Snippet: '{DOCS[doc_id]['text']}'")
            
    expanded_ids = set(seed_ids)
    frontier = set(seed_ids)
    
    for hop in range(expand_hops):
        if verbose:
            print(f"\n[Step 2: Following Connections - Hop {hop + 1}] Traversing the graph to find other documents sharing the exact same entities...")
        neighbours = set()
        for doc_id in frontier:
            doc_neighbors = doc_graph.get(doc_id, set())
            if verbose and doc_neighbors:
                doc_ents = set(doc_to_entities.get(doc_id, []))
                for n_id in doc_neighbors - expanded_ids:
                    n_ents = set(doc_to_entities.get(n_id, []))
                    shared = doc_ents.intersection(n_ents)
                    if shared:
                        print(f"    ↳ Discovered new [Doc {n_id}] because it perfectly shares entities {list(shared)} with seed [Doc {doc_id}]")
            neighbours |= doc_neighbors
        
        new_nodes = neighbours - expanded_ids
        expanded_ids |= new_nodes
        frontier = new_nodes

    if verbose:
        print(f"\n[Step 3: Re-ranking] Semantically re-scoring all {len(expanded_ids)} documents in the newly expanded pool directly against the original query to surface the absolute best actual answers...")

    q_emb = model.encode([query], normalize_embeddings=True).astype("float32")
    expanded_list = sorted(expanded_ids)
    exp_embs = corpus_embeddings[expanded_list]
    
    scores = (q_emb @ exp_embs.T).flatten()
    ranked = sorted(zip(expanded_list, scores.tolist()), key=lambda x: x[1], reverse=True)
    
    return ranked[:top_k]


## 9. Compare Results (GraphRAG vs All Strategies)
Visualizing how GraphRAG stacks up against all previous retrieval methodologies head-to-head.

In [9]:
print(f'QUERY: "{input_query}"')
print_results("Keyword only", lambda: keyword_search(query=input_query, top_k=3))
print_results("Semantic only", lambda: semantic_search(query=input_query, top_k=3))
print_results("Hybrid — RRF (k=60)", lambda: hybrid_search(query=input_query, top_k=3, fusion="rrf"))
print_results("Hybrid — Convex α=0.5", lambda: hybrid_search(query=input_query, top_k=3, fusion="convex", alpha=0.5))
print_results("GraphRAG", lambda: graph_rag_search(input_query, top_k=3, expand_hops=1, verbose=True))

QUERY: "Why are the funds for a check still in PENDING_KB200 status?"

────────────────────────────────────────────────────────────
  Keyword only
────────────────────────────────────────────────────────────
  [04] score=4.1156  |  If details shows PENDING_KB200 this means clearance is still in progress.
  [45] score=3.3460  |  The new terminal update requires a reboot; code PENDING_KB200 indicates the patch is stalled waiting for admin to check it.
  [46] score=3.2302  |  Status PENDING_KB200 check tray means the paper jam is cleared but the tray sensor has not yet reset.

────────────────────────────────────────────────────────────
  Semantic only
────────────────────────────────────────────────────────────
  [31] score=0.6846  |  Status: PENDING_KB105 means a cash deposit was made at an ATM but has not yet been physically verified by a teller.
  [64] score=0.6638  |  Funds held from a pending check will immediately reflect in your current balance but not in your available balance.
 

## 10. Agentic RAG loop
Binding the hybrid search engine into an external loop acting as a tool for the Claude LLM to explore and answer autonomously.

In [10]:
# ── Tool definition exposed to Claude ───────────────────────────────────────
HYBRID_SEARCH_TOOL = {
    "name": "hybrid_search",
    "description": (
        "Search the document corpus using hybrid keyword + semantic retrieval. "
        "Use alpha close to 0 for exact keyword queries (error codes, IDs), "
        "alpha close to 1 for semantic/conceptual queries, and alpha=0.5 for mixed queries."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query. Rephrase for better retrieval if needed.",
            },
            "top_k": {
                "type": "integer",
                "description": "Number of documents to return (default 3, max 5).",
                "default": 3,
            },
            "alpha": {
                "type": "number",
                "description": "Weight for semantic (dense) retrieval. 0=Keyword only, 1=Semantic only, 0.5=balanced.",
                "default": 0.5,
            },
        },
        "required": ["query"],
    },
}

def execute_tool(tool_name: str, tool_input: dict) -> str:
    """Dispatch tool calls from the LLM to our local hybrid search."""
    if tool_name == "hybrid_search":
        query = tool_input["query"]
        top_k = min(int(tool_input.get("top_k", 3)), 5)
        alpha = float(tool_input.get("alpha", 0.5))
        results = hybrid_search(query, top_k=top_k, fusion="convex", alpha=alpha)
        hits = [
            {"doc_id": doc_id, "score": round(score, 4), "text": DOCS[doc_id]["text"]}
            for doc_id, score in results
        ]
        return json.dumps({"results": hits}, indent=2)
    return json.dumps({"error": f"Unknown tool: {tool_name}"})

def agentic_rag(user_question: str, max_iterations: int = 5, verbose: bool = True) -> str:
    client = anthropic.Anthropic()
    
    # The system prompt instructs the LLM on how to autonomously adjust the query and alpha
    # in subsequent iterations if the initial search results are insufficient.
    system_prompt = (
        "You are a helpful assistant with access to a document corpus via the hybrid_search tool. "
        "Before answering, always search for relevant documents. "
        "If the initial results are insufficient, search again with a different query or alpha value. "
        "Choose alpha carefully: use low alpha (≈0.1) for exact keyword lookups, "
        "high alpha (≈0.9) for conceptual questions, and 0.5 for general queries."
    )
    
    # Start the conversation history with the user's initial question
    conversation_history = [{"role": "user", "content": user_question}]
    iteration = 0
    
    while iteration < max_iterations:
        iteration += 1
        
        # The LLM determines the next action: either answering the user or invoking the search tool.
        # It generates a new query autonomously based on the system prompt and previous conversation history.
        response = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=1024,
            system=system_prompt,
            tools=[HYBRID_SEARCH_TOOL],
            messages=conversation_history,
        )
        
        if verbose:
            print(f"\n[Iteration {iteration}] stop_reason={response.stop_reason}")
            
        # Append the LLM's response (which contains its reasoning and tool requests) to the history
        conversation_history.append({"role": "assistant", "content": response.content})
        
        if response.stop_reason == "end_turn":
            # The LLM has gathered enough information and provided a final answer
            for block in response.content:
                if hasattr(block, "text"):
                    return block.text
            return "(no text response)"
            
        if response.stop_reason == "tool_use":
            # The LLM decided to search. We collect the results and feed them back in the next iteration.
            tool_calls = [b for b in response.content if b.type == "tool_use"]
            if verbose:
                print(f"  LLM decided to run {len(tool_calls)} queries in parallel:")
                for block in tool_calls:
                    print(f"    - {block.name}({json.dumps(block.input)})")
                    
            tool_results = []
            for block in tool_calls:
                result = execute_tool(block.name, block.input)
                if verbose:
                    print(f"  Tool result for {json.dumps(block.input)}:\n    {result[:300]}...")
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result,
                })
            # Add the search results back to the conversation history so the LLM can evaluate them
            conversation_history.append({"role": "user", "content": tool_results})
        else:
            break
            
    return "(max iterations reached)"

## 11. Run Agentic RAG
Executing the Agentic search pattern and stepping through the iterations of autonomous retrieval queries.

In [11]:
if os.environ.get("ANTHROPIC_API_KEY"):
    print("API Key found! Running REAL Agentic RAG...\n")
    final_answer = agentic_rag(user_question=input_query, max_iterations=5, verbose=True)
    print("\n── Agentic RAG Final answer ──")
    print(final_answer)
else:
    print("No ANTHROPIC_API_KEY found. Please set your Anthropic API Key in your environment variables to run the real Agentic RAG loop.")

API Key found! Running REAL Agentic RAG...


[Iteration 1] stop_reason=tool_use
  LLM decided to run 2 queries in parallel:
    - hybrid_search({"query": "PENDING_KB200 status", "alpha": 0.1})
    - hybrid_search({"query": "check funds pending status reasons", "alpha": 0.7})
  Tool result for {"query": "PENDING_KB200 status", "alpha": 0.1}:
    {
  "results": [
    {
      "doc_id": 4,
      "score": 1.0,
      "text": "If details shows PENDING_KB200 this means clearance is still in progress."
    },
    {
      "doc_id": 45,
      "score": 0.1009,
      "text": "The new terminal update requires a reboot; code PENDING_KB200 indicates the p...
  Tool result for {"query": "check funds pending status reasons", "alpha": 0.7}:
    {
  "results": [
    {
      "doc_id": 59,
      "score": 0.8326,
      "text": "If a check's status is still pending, the associated funds absolutely cannot be withdrawn yet."
    },
    {
      "doc_id": 64,
      "score": 0.5661,
      "text": "Funds held from 

## 12. Compare results from all Agentic queries
Manually inspecting the difference between standard one-shot search algorithms and the exact queries formulated by the LLM reasoning loop.

In [12]:
print(f'ORIGINAL QUERY: "{input_query}"')
print_results("Keyword only", lambda: keyword_search(query=input_query, top_k=3))
print_results("Semantic only", lambda: semantic_search(query=input_query, top_k=3))
print_results("Hybrid — RRF (k=60)", lambda: hybrid_search(query=input_query, top_k=3, fusion="rrf"))
print_results("Hybrid — Convex α=0.5", lambda: hybrid_search(query=input_query, top_k=3, fusion="convex", alpha=0.5))
print_results("GraphRAG", lambda: graph_rag_search(input_query, top_k=3, expand_hops=1, verbose=False))

print("\n" + "="*80)
print("  AGENTIC QUERIES GENERATED BY LLM (from step 11)")
print("="*80)
print_results('Agent Iteration 1 — "PENDING_KB200 status" (α=0.1)', lambda: hybrid_search(query="PENDING_KB200 status", top_k=3, fusion="convex", alpha=0.1))
print_results('Agent Iteration 2 — "check funds pending status reasons" (α=0.5)', lambda: hybrid_search(query="check funds pending status reasons", top_k=3, fusion="convex", alpha=0.5))
print_results('Agent Iteration 3 — "check clearance in progress pending funds" (α=0.6)', lambda: hybrid_search(query="check clearance in progress pending funds", top_k=3, fusion="convex", alpha=0.6))

ORIGINAL QUERY: "Why are the funds for a check still in PENDING_KB200 status?"

────────────────────────────────────────────────────────────
  Keyword only
────────────────────────────────────────────────────────────
  [04] score=4.1156  |  If details shows PENDING_KB200 this means clearance is still in progress.
  [45] score=3.3460  |  The new terminal update requires a reboot; code PENDING_KB200 indicates the patch is stalled waiting for admin to check it.
  [46] score=3.2302  |  Status PENDING_KB200 check tray means the paper jam is cleared but the tray sensor has not yet reset.

────────────────────────────────────────────────────────────
  Semantic only
────────────────────────────────────────────────────────────
  [31] score=0.6846  |  Status: PENDING_KB105 means a cash deposit was made at an ATM but has not yet been physically verified by a teller.
  [64] score=0.6638  |  Funds held from a pending check will immediately reflect in your current balance but not in your available b

## 13. Summary

**Decision Guide — Which Strategy to Use?**

| Methodology               | When Recommended                                           |
|---------------------------|------------------------------------------------------------|
| **Keyword Search**        | Exact keywords and especially, uncommon words in query     |
| **Semantic Search**       | Text with unknown content by meaning                       |
| **Hybrid Search**         | When both the above are needed                             |
| **GraphRAG**              | Graph connections needed                                   |
| **Agentic RAG**           | Versatile search, any scenario, multiple iterations needed |